# **Analisis Sentimen dengan Deep Learning**

# 0.Import Library

In [1]:
import random
import numpy as np
import pandas as pd
import torch
from torch import optim
import torch.nn.functional as F
from tqdm import tqdm
 
from transformers import BertForSequenceClassification, BertConfig, BertTokenizer
from nltk.tokenize import TweetTokenizer
 
from indonlu.utils.forward_fn import forward_sequence_classification
from indonlu.utils.metrics import document_sentiment_metrics_fn
from indonlu.utils.data_utils import DocumentSentimentDataset, DocumentSentimentDataLoader

d:\Machine Learning\NLP Project\Analisis Sentimen Deep Learning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Mmebuat fungsi untuk engatur dan menetapkan random seed
def set_seed(seed):
    random.seed(seed) 
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

# Membuat fungsi untuk menghitung jumlah parameter dalam model
def count_param(module, trainable=False):
    if trainable:
        return sum(p.numel() for p in module.parameters() if p.requires_grad)
    else:
        return sum(p.numel() for p in module.parameters())

# Membuat fungsi untuk mengatur learning rate
def get_lr(optimizer):
    for param_group in optimizer.param_groups:
        return param_group['lr']

# Membuat fungsi untuk mengkonversi metriks kedalam string
def metrics_to_string(metric_dict):
    string_list = []
    for key, value in metric_dict.items():
        string_list.append('{}:{:.2f}'.format(key, value))

    return ' '.join(string_list)

In [3]:
# Set random seed
set_seed(6022004) # Bebas mau diisi apa aja asalkan angka atau integer

# 1.Konfigurasi dan Load Pre-trained Model

In [4]:
# Load tokenizer dan config
tokenizer = BertTokenizer.from_pretrained('indobenchmark/indobert-base-p1')
config = BertConfig.from_pretrained('indobenchmark/indobert-base-p1')
config.num_labels = DocumentSentimentDataset.NUM_LABELS

# Instantiate model
model = BertForSequenceClassification.from_pretrained('indobenchmark/indobert-base-p1')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 35052.35it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [5]:
model

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(50000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,),

In [6]:
count_param(model)

124445189

# 2.Data Preprocessing

In [7]:
# Persiapan dataset untuk training
train_dataset_path = 'indonlu/dataset/smsa_doc-sentiment-prosa/train_preprocess.tsv'
valid_dataset_path = 'indonlu/dataset/smsa_doc-sentiment-prosa/valid_preprocess.tsv'
test_dataset_path = 'indonlu/dataset/smsa_doc-sentiment-prosa/test_preprocess_masked_label.tsv'

Di sini, kita akan menggunakan 2 kelas yang disediakan di PyTorch dalam modul torch.utils.data yaitu **Dataset** dan **DataLoader**. Disadur dari Pre-Trained Models for NLP Tasks Using PyTorch [39], kelas Dataset adalah sebuah abstract class yang perlu kita extend di PyTorch. Sedangkan, DataLoader adalah inti dari perangkat pemrosesan data di PyTorch. DataLoader menyediakan banyak fungsionalitas untuk mempersiapkan data termasuk berbagai metode sampling, komputasi paralel, dan pemrosesan terdistribusi. Nah, kita akan memindahkan objek dari kelas Dataset ke dalam objek dari kelas DataLoader untuk pemrosesan batch data lebih lanjut.

In [ ]:
# Secara umum kode Dataset dan DataLoader yang dibuat oleh pytorch seperti ini 
# Cell ini bukan untuk di run hanya untuk dibaca dan dipahami saja

class DocumentSentimentDataset(Dataset):
    # Static constant variabel 
    LABEL2INDEX = {'positive': 0, 'neutral': 1, 'negative': 2} # Map dari label string ke index
    INDEX2LABEL = {0: 'positive', 1: 'neutral', 2: 'negative'} # Map dari index ke label string
    NUM_LABELS = 3 # Jumlah label

    def load_data(self, path):
        df = pd.read_csv(path, sep='\t', header=None) # Baca tsv dengan pandas
        df.columns = ['text', 'sentiment'] # Berikan nama pada kolom tabel
        df['sentiment'] = df['sentiment'].apply(lambda x: self.LABEL2INDEX[x]) # Konversi string label ke index
        return df

    def __init__(self, dataset_path, tokenizer, *args, **kwargs):
        self.data = self.load_dataset(dataset_path) # Load tsv file

        # Assign tokenizer menggunakan tokenizer dengan subword dari huggingface
        self.tokenizer = tokenizer

    def __getitem__(self, index):
        data = self.data.loc[index, :] # Ambil data pada baris tertentu dari tabel
        text, sentiment = data['text'], data['sentiment'] # Ambil nilai text dan sentiment
        subwords = self.tokenizer.encoder(text) # Tokenisasi text menjadi subword

        # Return Numpy array dari subwords dan label
        return np.array(subwords), np.array(sentiment), data['text']

    def __len__(self):
        # Return panjang dari data
        return len(self.data)

In [ ]:
class DocumentSentimentDataLoader(DataLoader):
    def __init__(self, max_seq_len = 512, *args, **kwargs):
        super(DocumentSentimentDataLoader, self).__init__(*args, **kwargs)
        self.max_seq_len = max_seq_len # Assign batas maximum subwords
        self.collate_fn = self._collate_fn # Assign fugnsi collate_fn dengan fungsi yang didefinisikan

    def _collate_fn(self, batch):
        batch_size = len(batch) # Ambil batch size
        max_seq_len = max(map(lambda x: len(x[0]), batch)) # Cari panjang subword maksimal dari batch
        max_seq_len = min(self.max_seq_len, max_seq_len) # Bandingkan dengan batas yang sudah ditetapkan sebelumnya 

        # Buat buffer untuk subword, mask dan sentiment labels, inisialisasikan semuanya dengan 0
        subword_batch = np.zeros((batch_size, max_seq_len), dtype=np.int64)
        mask_batch = np.zeros((batch_size, max_seq_len), dtype=np.float32)
        sentiment_batch = np.zeros((batch_size, 1), dtype=np.int64)

        # Isi semua buffer
        for i, (subwords, sentiment, raw_seq) in enumerate(batch):
            subwords = subwords[:max_seq_len]
            subword_batch[i, :len(subwords)] = subwords
            mask_batch[i, :len(subwords)] = 1
            sentiment_batch[i, 0] = sentiment

        # Return subword, mask, dan sentiment data
        return subword_batch, mask_batch, sentiment_batch

In [20]:
train_dataset = DocumentSentimentDataset(train_dataset_path, tokenizer, lowercase=True)
valid_dataset = DocumentSentimentDataset(valid_dataset_path, tokenizer, lowercase=True)
test_dataset = DocumentSentimentDataset(test_dataset_path, tokenizer, lowercase=True)
 
train_loader = DocumentSentimentDataLoader(dataset=train_dataset, max_seq_len=512, batch_size=32, num_workers=0, shuffle=True)  
valid_loader = DocumentSentimentDataLoader(dataset=valid_dataset, max_seq_len=512, batch_size=32, num_workers=0, shuffle=False)  
test_loader = DocumentSentimentDataLoader(dataset=test_dataset, max_seq_len=512, batch_size=32, num_workers=0, shuffle=False)

In [21]:
print(train_dataset[0])

(array([    2,  6540,    92,  2970,   213,  4259,  3553,   899,    34,
         259,  5590,   262,  2558,   386,   899,  1687,    26,  1574,
       30470,   899,  3310, 30468, 22130, 30360,  6123,  6368, 30468,
       22130, 30360,  2652,  1746, 30468,  8869,  6540,    34,  6315,
        1622,  1256,  8949,   899, 30468,  4222,  1622,   752,   245,
         295,  2083, 30470,  2346,  7107,   300, 30470,   405,   724,
        5189, 30470,   843, 17464,   899,   540, 10989,  3331,  1107,
       30468,   119,  3221,    79,    34,  2170,    98,  9167, 30457,
           3]), array(0), 'warung ini dimiliki oleh pengusaha pabrik tahu yang sudah puluhan tahun terkenal membuat tahu putih di bandung . tahu berkualitas , dipadu keahlian memasak , dipadu kretivitas , jadilah warung yang menyajikan menu utama berbahan tahu , ditambah menu umum lain seperti ayam . semuanya selera indonesia . harga cukup terjangkau . jangan lewatkan tahu bletoka nya , tidak kalah dengan yang asli dari tegal !')


In [22]:
w2i, i2w = DocumentSentimentDataset.LABEL2INDEX, DocumentSentimentDataset.INDEX2LABEL
print(w2i)

{'positive': 0, 'neutral': 1, 'negative': 2}


In [23]:
print(i2w)

{0: 'positive', 1: 'neutral', 2: 'negative'}


# 3.Uji Model

In [26]:
text = 'Bahagia hatiku melihat pernikahan putri sulungku yang cantik jelita'
subwords = tokenizer.encode(text)
subwords = torch.LongTensor(subwords).view(1, -1).to(model.device)

logits = model(subwords)[0]
label = torch.topk(logits, k=1, dim=-1)[1].squeeze().item()

print(f'Text : {text} | Label: {i2w[label]} ({F.softmax(logits, dim=-1).squeeze()[label] * 100:.3f}%)')

Text : Bahagia hatiku melihat pernikahan putri sulungku yang cantik jelita | Label: negative (38.501%)


# 4.Fine-Tuning

In [27]:
# Pake cuda supaya proses pelatihan cepat (RTX 3050)
# Cek dulu cuda nya apakah terdeteksi atau tidak

import torch

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.14.0+cu130
True
NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [28]:
optimizer = optim.Adam(model.parameters(), lr=3e-6)
model = model.cuda()

In [33]:
# Train model
n_epoch = 5
for ep in range(n_epoch):
    model.train()
    torch.set_grad_enabled(True)

    total_train_loss = 0
    list_hyp, list_label = [], []

    train_pbar = tqdm(train_loader, leave=True, total=len(train_loader))
    for i, batch_data in enumerate(train_pbar):
        # Forward model
        loss, batch_hyp, batch_label = forward_sequence_classification(model, batch_data[:-1], i2w=i2w, device='cuda')

        # Update Model
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Kalkulasi total loss
        tr_loss = loss.item() # Mengambil data train loss 
        total_train_loss += tr_loss # Memasukan data train loss ke total_train_loss

        # Kalkulasikan metrik 
        list_hyp += batch_hyp
        list_label += batch_label
        train_pbar.set_description('(Epoch {}) Train Loss: {:.4f} Lr: {:.8f}'.format((ep + 1), total_train_loss / (i + 1), get_lr(optimizer)))

    # Kalkulasi metrik training
    metrics = document_sentiment_metrics_fn(list_hyp, list_label)
    print('(Epoch {}) Train Loss: {:.4f} {} Lr: {:.8f}'.format((ep + 1), total_train_loss / (i + 1), metrics_to_string(metrics), get_lr(optimizer)))

    # Evaluasi model dengan validasi
    model.eval()
    torch.set_grad_enabled(False)

    total_loss, total_correct, total_labels = 0, 0, 0
    list_hyp, list_label = [], []

    total_loss = 0
    pbar = tqdm(valid_loader, leave=True, total=len(valid_loader))
    for i, batch_data in enumerate(pbar):
        batch_seq = batch_data[-1]
        loss, batch_hyp, batch_label = forward_sequence_classification(model, batch_data[:-1], i2w=i2w, device='cuda')

        # Kalkulasi total loss
        valid_loss = loss.item() # Mengambil data validation loss
        total_loss += valid_loss # Memasukan data validation loss ke total loss

        # Kalkulasi metrik evaluasi
        list_hyp += batch_hyp
        list_label += batch_label
        metrics = document_sentiment_metrics_fn(list_hyp, list_label)

        pbar.set_description('Valid Loss: {:.4f} {}'.format(total_loss / (i + 1), metrics_to_string(metrics)))

    metrics = document_sentiment_metrics_fn(list_hyp, list_label)
    print('(Epoch {}) Valid Loss: {:.4f} {}'.format((ep + 1), total_loss / (i + 1), metrics_to_string(metrics)))


(Epoch 1) Train Loss: 0.0714 Lr: 0.00000300: 100%|██████████| 344/344 [02:47<00:00,  2.05it/s]


(Epoch 1) Train Loss: 0.0714 ACC:0.98 F1:0.98 REC:0.97 PRE:0.98 Lr: 0.00000300


Valid Loss: 0.1921 ACC:0.94 F1:0.91 REC:0.90 PRE:0.93: 100%|██████████| 40/40 [00:06<00:00,  5.97it/s]


(Epoch 1) Valid Loss: 0.1921 ACC:0.94 F1:0.91 REC:0.90 PRE:0.93


(Epoch 2) Train Loss: 0.0531 Lr: 0.00000300: 100%|██████████| 344/344 [02:50<00:00,  2.01it/s]


...

(Epoch 4) Valid Loss: 0.2377 ACC:0.94 F1:0.92 REC:0.91 PRE:0.93


(Epoch 5) Train Loss: 0.0199 Lr: 0.00000300: 100%|██████████| 344/344 [02:49<00:00,  2.02it/s]


(Epoch 5) Train Loss: 0.0199 ACC:1.00 F1:0.99 REC:0.99 PRE:1.00 Lr: 0.00000300


Valid Loss: 0.2486 ACC:0.94 F1:0.91 REC:0.90 PRE:0.92: 100%|██████████| 40/40 [00:06<00:00,  5.94it/s]

(Epoch 5) Valid Loss: 0.2486 ACC:0.94 F1:0.91 REC:0.90 PRE:0.92


In [34]:
# Evaluate on test

model.eval()
torch.set_grad_enabled(False)

total_loss, total_correct, total_labels = 0, 0, 0
list_hyp, list_label = [], []

pbar = tqdm(test_loader, leave=True, total=len(test_loader))
for i, batch_data in enumerate(pbar):
    _, batch_hyp, _ = forward_sequence_classification(
        model, batch_data[:-1], i2w=i2w, device="cuda"
    )
    list_hyp += batch_hyp

# Save prediction
df = pd.DataFrame({"label": list_hyp}).reset_index()
df.to_csv("pred.txt", index=False)

print(df)

100%|██████████| 16/16 [00:01<00:00,  9.39it/s]

     index     label
0        0  negative
1        1  negative
2        2  negative
3        3  negative
4        4  negative
..     ...       ...
495    495   neutral
496    496   neutral
497    497  positive
498    498  positive
499    499  positive

[500 rows x 2 columns]


In [35]:
# Perdiksi sentimen
text = 'Bahagia hatiku melihat pernikahan putri sulungku yang cantik jelita'
subwords = tokenizer.encode(text)
subwords = torch.LongTensor(subwords).view(1, -1).to(model.device)
 
logits = model(subwords)[0]
label = torch.topk(logits, k=1, dim=-1)[1].squeeze().item()
 
print(f'Text: {text} | Label : {i2w[label]} ({F.softmax(logits, dim=-1).squeeze()[label] * 100:.3f}%)')

Text: Bahagia hatiku melihat pernikahan putri sulungku yang cantik jelita | Label : positive (99.879%)


In [36]:
def predict_sentiment(text):
    subwords = tokenizer.encode(text)
    subwords = torch.LongTensor(subwords).view(1, -1).to(model.device)

    logits = model(subwords)[0]
    label = torch.topk(logits, k=1, dim=-1)[1].squeeze().item()

    print(
        f"Text: {text} | Label : {i2w[label]} ({F.softmax(logits, dim=-1).squeeze()[label] * 100:.3f}%)"
    )

In [37]:
predict_sentiment('Ronaldo pergi ke Mall Grand Indonesia membeli cilok')

Text: Ronaldo pergi ke Mall Grand Indonesia membeli cilok | Label : neutral (99.764%)


In [38]:
predict_sentiment('Sayang, aku marah')

Text: Sayang, aku marah | Label : negative (99.837%)


In [39]:
predict_sentiment('Merasa kagum dengan toko ini tapi berubah menjadi kecewa setelah transaksi')

Text: Merasa kagum dengan toko ini tapi berubah menjadi kecewa setelah transaksi | Label : negative (99.864%)
